# Extracción de características y balanceo

Vamos a extraer características para entrenamiento y balancear el dataset

In [28]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

In [29]:
df = pd.read_csv('./data/cleaned.csv')

# Construir timestamp
df['timestamp'] = (
    pd.to_datetime('2024' + df['DE13_local_date'].astype(str).str.zfill(4), format='%Y%m%d')
    + pd.to_timedelta(df['hour_local'], unit='h')
)

# Ordenar por cliente y tiempo
df = df.sort_values(['client_id', 'timestamp']).reset_index(drop=True)

# df_rolling: version con timestamp como indice, para las ventanas deslizantes
df_rolling = df.set_index('timestamp')

In [30]:
df.shape

(100003, 39)

In [31]:
# Minutos desde la ultima transaccion del cliente. Velocidad anomala = posible fraude.
df['time_since_last_txn_min'] = (
    df.groupby('client_id')['timestamp']
    .diff()
    .dt.total_seconds()
    .div(60)
    .fillna(9999)
)

In [32]:
hourly = (
    df.groupby(['client_id', 'timestamp'])
    .agg(
        txn_in_hour=('transaction_id', 'count'),
        amount_in_hour=('amount_usd', 'sum')
    )
    .reset_index()
)

In [33]:
hourly = hourly.sort_values(
    ['client_id', 'timestamp']
).reset_index(drop=True)

In [34]:
# Cantidad de transacciones del cliente en la ultima hora.
hourly['txn_count_last_1h'] = (
    hourly.groupby('client_id')
    .rolling(
        '1h',
        on='timestamp',
        closed='left'
    )['txn_in_hour']
    .sum()
    .reset_index(drop=True)
)

In [35]:
# Cantidad de transacciones del cliente en las ultimas 24 horas.
hourly['txn_count_last_24h'] = (
    hourly.groupby('client_id')
    .rolling(
        '24h',
        on='timestamp',
        closed='left'
    )['txn_in_hour']
    .sum()
    .reset_index(drop=True)
)

In [36]:
# 4. amount_sum_last_1h
# Muchas transacciones pequeñas rapidas es patron de fraude.
hourly['amount_sum_last_1h'] = (
    hourly.groupby('client_id')
    .rolling(
        '1h',
        on='timestamp',
        closed='left'
    )['amount_in_hour']
    .sum()
    .reset_index(drop=True)
)

In [37]:
# Monto total gastado por el cliente en las ultimas 24 horas.

hourly['amount_sum_last_24h'] = (
    hourly.groupby('client_id')
    .rolling(
        '24h',
        on='timestamp',
        closed='left'
    )['amount_in_hour']
    .sum()
    .reset_index(drop=True)
)

In [38]:
cols = [
    'txn_count_last_1h',
    'txn_count_last_24h',
    'amount_sum_last_1h',
    'amount_sum_last_24h'
]

hourly[cols] = hourly[cols].fillna(0)



df = df.merge(
    hourly[
        [
            'client_id',
            'timestamp',
            'txn_count_last_1h',
            'txn_count_last_24h',
            'amount_sum_last_1h',
            'amount_sum_last_24h'
        ]
    ],
    on=['client_id', 'timestamp'],
    how='left'
)

In [39]:
grouped = df.groupby('client_id')['amount_usd']

exp_mean = (
    grouped.expanding()
    .mean()
    .reset_index(level=0, drop=True)
)

exp_std = (
    grouped.expanding()
    .std()
    .reset_index(level=0, drop=True)
)

df['amount_zscore_customer'] = (
    (
        df['amount_usd'].to_numpy()
        - exp_mean.to_numpy()
    )
    /
    (exp_std.to_numpy() + 1e-8)
)

In [40]:

df['amount_ratio_vs_baseline'] = df['amount_usd'] / (df['client_baseline_amount'] + 1e-8)

In [41]:
#La transaccion ocurrio entre medianoche y las 5am.
df['is_night'] = df['hour_local'].between(0, 5).astype(int)

In [42]:
# Indica si el comercio electronico
df['is_high_risk_channel'] = (df['channel'] == 'ECOM').astype(int)

In [43]:
# Interaccion entre distancia del hogar y monto.
df['distance_x_amount'] = df['distance_from_home_km'] * df['amount_usd']

In [44]:
df['amount_zscore_customer'] = (
    df['amount_zscore_customer']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

In [45]:
# Verificacion
new_features = [
    'time_since_last_txn_min', 'txn_count_last_1h', 'txn_count_last_24h',
    'amount_sum_last_1h', 'amount_sum_last_24h', 'amount_zscore_customer',
    'amount_ratio_vs_baseline', 'is_night', 'is_high_risk_channel', 'distance_x_amount'
]

print(df[new_features].describe())
print("\nPromedio por clase:")
print(df[new_features + ['is_fraud']].groupby('is_fraud').mean())

       time_since_last_txn_min  txn_count_last_1h  txn_count_last_24h  \
count            100003.000000      100003.000000       100003.000000   
mean              10225.364839           0.012740            0.164955   
std               12898.379908           0.140632            0.474663   
min                   0.000000           0.000000            0.000000   
25%                2760.000000           0.000000            0.000000   
50%                7320.000000           0.000000            0.000000   
75%               13560.000000           0.000000            0.000000   
max              328380.000000           6.000000            9.000000   

       amount_sum_last_1h  amount_sum_last_24h  amount_zscore_customer  \
count       100003.000000        100003.000000           100003.000000   
mean             8.712411            91.339990                0.009734   
std            142.282405           499.071508                0.935416   
min              0.000000             0.000000

In [46]:

df['date'] = pd.to_datetime(
    '2025' + df['DE13_local_date'].astype(str).str.zfill(4),
    format='%Y%m%d'
)

train_df = df[df['date'] < '2025-06-01'].copy()

test_df = df[df['date'] >= '2025-06-01'].copy()

print(train_df.shape)
print(test_df.shape)

print(train_df['date'].min(), train_df['date'].max())
print(test_df['date'].min(), test_df['date'].max())

(83836, 50)
(16167, 50)
2025-01-01 00:00:00 2025-05-31 00:00:00
2025-06-01 00:00:00 2025-12-31 00:00:00


In [47]:
cols_to_drop = [
    'is_fraud',
    'client_id',      
    'transaction_id', 
    'timestamp',      
    'channel',
    'DE13_local_date',
    'DE14_expiration_date',
    'DE15_settlement_date', 
    'Unnamed: 0',
    'date'      
]

In [48]:
train_df['is_fraud'].value_counts()

is_fraud
0.0    79645
1.0     4191
Name: count, dtype: int64

In [50]:
test_df['is_fraud'].value_counts()

is_fraud
0.0    15439
1.0      728
Name: count, dtype: int64

In [51]:
X = train_df.drop(cols_to_drop, axis=1)
y = train_df['is_fraud']

In [52]:
cat_cols = X.select_dtypes(include='object').columns.tolist()
print("Categóricas restantes:", cat_cols)  # verificar qué queda

for col in cat_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

Categóricas restantes: ['client_segment', 'card_brand', 'DE52_pin_data_present', 'DE55_emv_data_present', 'DE60_pos_terminal_type', 'DE123_pos_data_code', 'currency_tx_alpha', 'day_of_week', 'client_home_city']


/tmp/ipykernel_52947/1058399875.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns.tolist()


In [53]:
pipeline = Pipeline([
    ('oversample', ADASYN(
        sampling_strategy={1: 8382},
        random_state=42,
        n_neighbors=5
    )),
    ('undersample', RandomUnderSampler(
        sampling_strategy={0: 50292},
        random_state=42
    ))
])

X_resampled, y_resampled = pipeline.fit_resample(X, y)

df_balanced = X_resampled.copy()
df_balanced['is_fraud'] = y_resampled

print(Counter(y_resampled))
print(df_balanced.shape)

Counter({0.0: 50292, 1.0: 8349})
(58641, 41)


In [54]:
# Guardar resultado
df_balanced.to_csv('./data/balanced_training.csv', index=False)
print("Archivo guardado como generacion_vars.csv")

Archivo guardado como generacion_vars.csv


In [56]:
X_test= test_df.drop(cols_to_drop, axis=1)
df_balanced_test = X_test.copy()



In [57]:
df_balanced_test.to_csv('./data/balanced_test.csv', index=False)